# Consolidação final do SICOR Cerrado — 2021 a 2025

Este notebook recria o consolidado diretamente a partir dos cinco arquivos anuais:

- `sicor_cerrado_2021.csv`
- `sicor_cerrado_2022.csv`
- `sicor_cerrado_2023.csv`
- `sicor_cerrado_2024.csv`
- `sicor_cerrado_2025.csv`

A substituição do arquivo consolidado só acontece depois que todos os anos forem processados e validados.

## Totais anuais esperados

| Ano | Linhas |
|---:|---:|
| 2021 | 39.013 |
| 2022 | 40.753 |
| 2023 | 93.911 |
| 2024 | 105.194 |
| 2025 | 98.594 |
| **Total** | **377.465** |

Ao final, o notebook:

- mostra todas as colunas;
- mostra o `head(5)`;
- informa a quantidade de linhas;
- apresenta os registros por ano;
- verifica `REF_BACEN` únicos e repetidos.

## Correção V2

Nesta versão, os totais esperados não são fixados manualmente. O notebook conta os registros existentes em cada arquivo anual no início da execução e usa esses valores para validar o arquivo consolidado.

Isso evita falhas causadas por números antigos, arquivos anuais recriados ou diferenças entre execuções.

In [7]:
# ============================================================
# 1. MONTAR O GOOGLE DRIVE E IMPORTAR BIBLIOTECAS
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import Counter
import gc
import os
import shutil

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.expand_frame_repr", False)

print("Ambiente preparado.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ambiente preparado.


In [8]:
# ============================================================
# 2. CAMINHOS E PARÂMETROS
# ============================================================

PASTA = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/segundo/"
    "sicor_cerrado_2021_2025"
)

# "G:\Meu Drive\Colab Notebooks\fluxo\Envio\Base\Assentamento Brasil.shx"

ANOS = [2021, 2022, 2023, 2024, 2025]

ARQUIVOS_ANUAIS = {
    ano: PASTA / f"sicor_cerrado_{ano}.csv"
    for ano in ANOS
}

ARQUIVO_FINAL = PASTA / "sicor_cerrado_2021_2025.csv"
ARQUIVO_TEMPORARIO = PASTA / "sicor_cerrado_2021_2025_TEMP.csv"
ARQUIVO_BACKUP = PASTA / "sicor_cerrado_2021_2025_BACKUP_INCOMPLETO.csv"

SEPARADOR = ";"
TAMANHO_BLOCO = 50_000
CODIFICACOES = ["utf-8-sig", "utf-8", "latin-1", "cp1252"]

# Os totais esperados serão calculados automaticamente a partir dos
# cinco arquivos anuais. Não são usados valores digitados manualmente.

print("Pasta de trabalho:", PASTA)


Pasta de trabalho: /content/drive/MyDrive/Colab Notebooks/fluxo/Envio/segundo/sicor_cerrado_2021_2025


In [9]:
# ============================================================
# 3. VERIFICAR OS CINCO ARQUIVOS ANUAIS
# ============================================================

for ano, caminho in ARQUIVOS_ANUAIS.items():
    if not caminho.exists():
        raise FileNotFoundError(
            f"O arquivo anual de {ano} não foi encontrado:\n{caminho}"
        )

    tamanho_mb = caminho.stat().st_size / (1024 ** 2)
    print(f"{ano}: OK | {tamanho_mb:,.2f} MB | {caminho.name}")

print("\nTodos os arquivos anuais foram encontrados.")

2021: OK | 52.81 MB | sicor_cerrado_2021.csv
2022: OK | 47.78 MB | sicor_cerrado_2022.csv
2023: OK | 63.29 MB | sicor_cerrado_2023.csv
2024: OK | 67.27 MB | sicor_cerrado_2024.csv
2025: OK | 85.01 MB | sicor_cerrado_2025.csv

Todos os arquivos anuais foram encontrados.


In [10]:
# ============================================================
# 4. DETECTAR CODIFICAÇÃO, LER CABEÇALHOS E CONTAR AS FONTES
# ============================================================

def detectar_codificacao_e_colunas(caminho):
    ultimo_erro = None

    for codificacao in CODIFICACOES:
        try:
            teste = pd.read_csv(
                caminho,
                sep=SEPARADOR,
                encoding=codificacao,
                nrows=5,
                low_memory=False
            )

            if len(teste.columns) <= 1:
                continue

            colunas = [
                str(coluna).lstrip("\ufeff")
                for coluna in teste.columns
            ]

            return codificacao, colunas

        except Exception as erro:
            ultimo_erro = erro

    raise RuntimeError(
        f"Não foi possível ler {caminho.name}. "
        f"Último erro: {ultimo_erro}"
    )


def contar_linhas_csv(caminho, codificacao, primeira_coluna):
    """
    Conta as linhas usando apenas a primeira coluna.
    Isso reduz o uso de memória e cria uma referência independente
    para validar a consolidação.
    """
    total = 0

    leitor = pd.read_csv(
        caminho,
        sep=SEPARADOR,
        encoding=codificacao,
        usecols=[primeira_coluna],
        chunksize=TAMANHO_BLOCO,
        low_memory=False
    )

    for bloco in leitor:
        total += len(bloco)

    return total


METADADOS = {}
COLUNAS_CONSOLIDADAS = []
TOTAIS_ESPERADOS = {}

for ano in ANOS:
    caminho = ARQUIVOS_ANUAIS[ano]
    codificacao, colunas = detectar_codificacao_e_colunas(caminho)

    METADADOS[ano] = {
        "codificacao": codificacao,
        "colunas": colunas
    }

    for coluna in colunas:
        if coluna not in COLUNAS_CONSOLIDADAS:
            COLUNAS_CONSOLIDADAS.append(coluna)

    primeira_coluna = colunas[0]

    total_fonte = contar_linhas_csv(
        caminho,
        codificacao,
        primeira_coluna
    )

    TOTAIS_ESPERADOS[ano] = total_fonte

    print(
        f"{ano}: {total_fonte:,} linhas na fonte | "
        f"{len(colunas)} colunas"
    )

if "ANO_BASE" not in COLUNAS_CONSOLIDADAS:
    COLUNAS_CONSOLIDADAS.append("ANO_BASE")

TOTAL_ESPERADO = sum(TOTAIS_ESPERADOS.values())

print("\nQuantidade de colunas consolidadas:", len(COLUNAS_CONSOLIDADAS))
print("Total esperado calculado das fontes:", f"{TOTAL_ESPERADO:,}")

print("\nLista completa de colunas:")
for numero, coluna in enumerate(COLUNAS_CONSOLIDADAS, start=1):
    print(f"{numero:03d} - {coluna}")


2021: 98,209 linhas na fonte | 5 colunas
2022: 84,771 linhas na fonte | 5 colunas
2023: 93,911 linhas na fonte | 5 colunas
2024: 105,194 linhas na fonte | 5 colunas
2025: 98,594 linhas na fonte | 5 colunas

Quantidade de colunas consolidadas: 5
Total esperado calculado das fontes: 480,679

Lista completa de colunas:
001 - #REF_BACEN
002 - NU_ORDEM
003 - NU_INDICE
004 - GT_GEOMETRIA
005 - ANO_BASE


In [11]:
# ============================================================
# 5. CRIAR O CONSOLIDADO EM ARQUIVO TEMPORÁRIO
# ============================================================

if ARQUIVO_TEMPORARIO.exists():
    ARQUIVO_TEMPORARIO.unlink()

cabecalho = True
totais_processados = {}
total_geral = 0

for ano in ANOS:
    caminho = ARQUIVOS_ANUAIS[ano]
    codificacao = METADADOS[ano]["codificacao"]
    total_ano = 0

    print("\n" + "=" * 80)
    print(f"PROCESSANDO {ano}")
    print("=" * 80)

    leitor = pd.read_csv(
        caminho,
        sep=SEPARADOR,
        encoding=codificacao,
        chunksize=TAMANHO_BLOCO,
        low_memory=False
    )

    for numero_bloco, bloco in enumerate(leitor, start=1):
        bloco.columns = [
            str(coluna).lstrip("\ufeff")
            for coluna in bloco.columns
        ]

        # Remove eventuais marcas BOM presentes no início de valores.
        primeira_coluna = bloco.columns[0]
        bloco[primeira_coluna] = bloco[primeira_coluna].map(
            lambda valor: valor.lstrip("\ufeff")
            if isinstance(valor, str)
            else valor
        )

        # Corrige o ano, independentemente do conteúdo anterior.
        bloco["ANO_BASE"] = ano

        # Acrescenta colunas ausentes no arquivo daquele ano.
        for coluna in COLUNAS_CONSOLIDADAS:
            if coluna not in bloco.columns:
                bloco[coluna] = pd.NA

        # Mantém a mesma ordem de colunas em todos os blocos.
        bloco = bloco.reindex(columns=COLUNAS_CONSOLIDADAS)

        codificacao_saida = "utf-8-sig" if cabecalho else "utf-8"

        bloco.to_csv(
            ARQUIVO_TEMPORARIO,
            mode="a",
            sep=SEPARADOR,
            encoding=codificacao_saida,
            index=False,
            header=cabecalho
        )

        quantidade = len(bloco)
        total_ano += quantidade
        total_geral += quantidade
        cabecalho = False

        print(
            f"Bloco {numero_bloco:03d}: "
            f"{quantidade:,} linhas | "
            f"acumulado do ano={total_ano:,}"
        )

        del bloco
        gc.collect()

    totais_processados[ano] = total_ano

    esperado = TOTAIS_ESPERADOS[ano]
    situacao = "OK" if total_ano == esperado else "DIVERGENTE"

    print(
        f"Total de {ano}: {total_ano:,} | "
        f"esperado={esperado:,} | {situacao}"
    )

print("\nTotal temporário:", f"{total_geral:,}")


PROCESSANDO 2021
Bloco 001: 50,000 linhas | acumulado do ano=50,000
Bloco 002: 48,209 linhas | acumulado do ano=98,209
Total de 2021: 98,209 | esperado=98,209 | OK

PROCESSANDO 2022
Bloco 001: 50,000 linhas | acumulado do ano=50,000
Bloco 002: 34,771 linhas | acumulado do ano=84,771
Total de 2022: 84,771 | esperado=84,771 | OK

PROCESSANDO 2023
Bloco 001: 50,000 linhas | acumulado do ano=50,000
Bloco 002: 43,911 linhas | acumulado do ano=93,911
Total de 2023: 93,911 | esperado=93,911 | OK

PROCESSANDO 2024
Bloco 001: 50,000 linhas | acumulado do ano=50,000
Bloco 002: 50,000 linhas | acumulado do ano=100,000
Bloco 003: 5,194 linhas | acumulado do ano=105,194
Total de 2024: 105,194 | esperado=105,194 | OK

PROCESSANDO 2025
Bloco 001: 50,000 linhas | acumulado do ano=50,000
Bloco 002: 48,594 linhas | acumulado do ano=98,594
Total de 2025: 98,594 | esperado=98,594 | OK

Total temporário: 480,679


In [12]:
# ============================================================
# 6. VALIDAR ANTES DE SUBSTITUIR O ARQUIVO FINAL
# ============================================================

df_validacao = pd.DataFrame(
    [
        {
            "ANO_BASE": ano,
            "LINHAS_NA_FONTE": TOTAIS_ESPERADOS[ano],
            "LINHAS_PROCESSADAS": totais_processados.get(ano, 0),
            "DIFERENCA": (
                totais_processados.get(ano, 0)
                - TOTAIS_ESPERADOS[ano]
            )
        }
        for ano in ANOS
    ]
)

print("VALIDAÇÃO POR ANO")
display(df_validacao)

divergencias = df_validacao[
    df_validacao["DIFERENCA"] != 0
].copy()

if not divergencias.empty:
    raise RuntimeError(
        "A consolidação foi interrompida porque pelo menos um arquivo "
        "anual não foi integralmente transferido para o temporário. "
        "Consulte a tabela acima para identificar o ano e a diferença."
    )

if total_geral != TOTAL_ESPERADO:
    raise RuntimeError(
        f"Total geral processado: {total_geral:,}. "
        f"Total calculado diretamente das fontes: {TOTAL_ESPERADO:,}."
    )

print("Validação anual aprovada.")
print("Validação total aprovada.")
print(f"Total confirmado: {total_geral:,} linhas.")


VALIDAÇÃO POR ANO


,ANO_BASE,LINHAS_NA_FONTE,LINHAS_PROCESSADAS,DIFERENCA
0,2021,98209,98209,0
1,2022,84771,84771,0
2,2023,93911,93911,0
3,2024,105194,105194,0
4,2025,98594,98594,0


Validação anual aprovada.
Validação total aprovada.
Total confirmado: 480,679 linhas.


In [13]:
# ============================================================
# 7. SUBSTITUIR O CONSOLIDADO SOMENTE APÓS A VALIDAÇÃO
# ============================================================

if ARQUIVO_BACKUP.exists():
    ARQUIVO_BACKUP.unlink()

if ARQUIVO_FINAL.exists():
    ARQUIVO_FINAL.rename(ARQUIVO_BACKUP)
    print("Arquivo anterior preservado como:")
    print(ARQUIVO_BACKUP)

ARQUIVO_TEMPORARIO.rename(ARQUIVO_FINAL)

print("\nNovo arquivo consolidado criado com sucesso:")
print(ARQUIVO_FINAL)
print(f"Tamanho: {ARQUIVO_FINAL.stat().st_size / (1024 ** 2):,.2f} MB")


Novo arquivo consolidado criado com sucesso:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/segundo/sicor_cerrado_2021_2025/sicor_cerrado_2021_2025.csv
Tamanho: 316.15 MB


In [14]:
# ============================================================
# 8. VALIDAR O ARQUIVO FINAL E MOSTRAR TODAS AS COLUNAS
# ============================================================

df_head = pd.read_csv(
    ARQUIVO_FINAL,
    sep=SEPARADOR,
    encoding="utf-8-sig",
    nrows=5,
    low_memory=False
)

print("Quantidade de colunas:", len(df_head.columns))
print("\nLISTA COMPLETA DE COLUNAS")
print("=" * 90)

for numero, coluna in enumerate(df_head.columns, start=1):
    print(f"{numero:03d} - {coluna}")

print("\nHEAD COM TODAS AS COLUNAS")
display(df_head)

Quantidade de colunas: 5

LISTA COMPLETA DE COLUNAS
001 - #REF_BACEN
002 - NU_ORDEM
003 - NU_INDICE
004 - GT_GEOMETRIA
005 - ANO_BASE

HEAD COM TODAS AS COLUNAS


,#REF_BACEN,NU_ORDEM,NU_INDICE,GT_GEOMETRIA,ANO_BASE
0,511405404,1,0,"POLYGON ((-47.521674 -18.527653 0,-47.521697 -18.528289 0,-47.519128 -18.528105 0,-47.519506 -18.526262 0,-47.519955...",2021
1,511405405,1,0,"POLYGON ((-47.579356 -18.906067 0,-47.578942 -18.906589 0,-47.578871 -18.907144 0,-47.578482 -18.907021 0,-47.57761 ...",2021
2,511405424,1,0,"POLYGON ((-44.441404 -16.832015 0,-44.441419 -16.831524 0,-44.440824 -16.831596 0,-44.439808999999897 -16.831699 0,-...",2021
3,511405446,1,0,"POLYGON ((-44.965694 -20.079657 0,-44.967058 -20.077494 0,-44.968458 -20.078419 0,-44.970597 -20.078106 0,-44.972607...",2021
4,511405448,1,0,"POLYGON ((-45.075154 -18.327891 0,-45.077628 -18.329547 0,-45.078869 -18.329854 0,-45.079659 -18.329899 0,-45.08048 ...",2021


In [15]:
# ============================================================
# 9. ANALISAR LINHAS E REF_BACEN NO ARQUIVO FINAL
# ============================================================

candidatas_ref = ["#REF_BACEN", "REF_BACEN", "CD_REF_BACEN"]

COLUNA_REF = next(
    (
        coluna
        for coluna in candidatas_ref
        if coluna in df_head.columns
    ),
    None
)

if COLUNA_REF is None:
    raise KeyError(
        "Nenhuma coluna REF_BACEN foi encontrada. "
        f"Procuradas: {candidatas_ref}"
    )

contador_ref = Counter()
contador_ano = Counter()
total_linhas_validacao = 0
total_ref_nulo = 0

leitor = pd.read_csv(
    ARQUIVO_FINAL,
    sep=SEPARADOR,
    encoding="utf-8-sig",
    usecols=[COLUNA_REF, "ANO_BASE"],
    dtype={COLUNA_REF: "string"},
    chunksize=TAMANHO_BLOCO,
    low_memory=False
)

for bloco in leitor:
    total_linhas_validacao += len(bloco)

    refs = bloco[COLUNA_REF].astype("string").str.strip()

    mascara_valida = (
        refs.notna()
        & refs.ne("")
        & refs.str.lower().ne("nan")
    )

    total_ref_nulo += int((~mascara_valida).sum())
    contador_ref.update(refs.loc[mascara_valida].tolist())

    contador_ano.update(
        bloco["ANO_BASE"]
        .dropna()
        .astype(int)
        .tolist()
    )

    del bloco, refs, mascara_valida
    gc.collect()

refs_repetidos = {
    ref: quantidade
    for ref, quantidade in contador_ref.items()
    if quantidade > 1
}

print("RESULTADOS FINAIS")
print("=" * 90)
print(f"Total de linhas: {total_linhas_validacao:,}")
print(f"REF_BACEN válidos: {sum(contador_ref.values()):,}")
print(f"REF_BACEN nulos ou vazios: {total_ref_nulo:,}")
print(f"REF_BACEN únicos: {len(contador_ref):,}")
print(
    "REF_BACEN distintos que aparecem mais de uma vez:",
    f"{len(refs_repetidos):,}"
)
print(
    "Linhas associadas a REF_BACEN repetidos:",
    f"{sum(refs_repetidos.values()):,}"
)
print(
    "Repetições excedentes:",
    f"{sum(qtd - 1 for qtd in refs_repetidos.values()):,}"
)

print("\nREGISTROS POR ANO")
for ano in sorted(contador_ano):
    print(f"{ano}: {contador_ano[ano]:,}")

RESULTADOS FINAIS
Total de linhas: 480,679
REF_BACEN válidos: 480,679
REF_BACEN nulos ou vazios: 0
REF_BACEN únicos: 334,166
REF_BACEN distintos que aparecem mais de uma vez: 70,261
Linhas associadas a REF_BACEN repetidos: 216,774
Repetições excedentes: 146,513

REGISTROS POR ANO
2021: 98,209
2022: 84,771
2023: 93,911
2024: 105,194
2025: 98,594


In [16]:
# ============================================================
# 10. MOSTRAR OS REF_BACEN MAIS REPETIDOS
# ============================================================

df_ref_frequencia = pd.DataFrame(
    contador_ref.most_common(),
    columns=[COLUNA_REF, "QUANTIDADE_DE_LINHAS"]
)

df_ref_repetidos = df_ref_frequencia[
    df_ref_frequencia["QUANTIDADE_DE_LINHAS"] > 1
].copy()

print("20 REF_BACEN mais frequentes:")
display(df_ref_repetidos.head(20))

20 REF_BACEN mais frequentes:


,#REF_BACEN,QUANTIDADE_DE_LINHAS
0,515991166,229
1,514010149,225
2,512053817,177
3,512970266,159
4,512053989,157
5,512970267,140
6,520215592,139
7,521050615,132
8,513464143,118
9,519704871,111


In [17]:
# ============================================================
# 11. CRIAR ID_GLEBA SEQUENCIAL PARA TODOS OS REGISTROS
# ============================================================

ARQUIVO_TEMP_ID = (
    PASTA / "sicor_cerrado_2021_2025_TEMP_ID_GLEBA.csv"
)

ARQUIVO_BACKUP_SEM_ID = (
    PASTA / "sicor_cerrado_2021_2025_SEM_ID_GLEBA.csv"
)

# Remove somente um arquivo temporário de execução anterior.
if ARQUIVO_TEMP_ID.exists():
    ARQUIVO_TEMP_ID.unlink()

proximo_id = 1
total_processado = 0
escrever_cabecalho = True

leitor = pd.read_csv(
    ARQUIVO_FINAL,
    sep=SEPARADOR,
    encoding="utf-8-sig",
    chunksize=TAMANHO_BLOCO,
    low_memory=False
)

for numero_bloco, bloco in enumerate(leitor, start=1):

    # Permite executar novamente a célula sem duplicar a coluna.
    if "ID_GLEBA" in bloco.columns:
        bloco = bloco.drop(columns=["ID_GLEBA"])

    quantidade_bloco = len(bloco)
    ultimo_id = proximo_id + quantidade_bloco

    # Insere ID_GLEBA como primeira coluna.
    bloco.insert(
        0,
        "ID_GLEBA",
        range(proximo_id, ultimo_id)
    )

    codificacao_saida = (
        "utf-8-sig"
        if escrever_cabecalho
        else "utf-8"
    )

    bloco.to_csv(
        ARQUIVO_TEMP_ID,
        mode="a",
        sep=SEPARADOR,
        encoding=codificacao_saida,
        index=False,
        header=escrever_cabecalho
    )

    total_processado += quantidade_bloco
    proximo_id = ultimo_id
    escrever_cabecalho = False

    print(
        f"Bloco {numero_bloco:03d}: "
        f"{quantidade_bloco:,} registros | "
        f"ID_GLEBA até {proximo_id - 1:,}"
    )

    del bloco
    gc.collect()

print("\nArquivo temporário concluído.")
print(f"Total de registros numerados: {total_processado:,}")

Bloco 001: 50,000 registros | ID_GLEBA até 50,000
Bloco 002: 50,000 registros | ID_GLEBA até 100,000
Bloco 003: 50,000 registros | ID_GLEBA até 150,000
Bloco 004: 50,000 registros | ID_GLEBA até 200,000
Bloco 005: 50,000 registros | ID_GLEBA até 250,000
Bloco 006: 50,000 registros | ID_GLEBA até 300,000
Bloco 007: 50,000 registros | ID_GLEBA até 350,000
Bloco 008: 50,000 registros | ID_GLEBA até 400,000
Bloco 009: 50,000 registros | ID_GLEBA até 450,000
Bloco 010: 30,679 registros | ID_GLEBA até 480,679

Arquivo temporário concluído.
Total de registros numerados: 480,679


In [18]:
# ============================================================
# VALIDAR A COLUNA ID_GLEBA
# ============================================================

total_validado = 0
id_esperado = 1

leitor_validacao = pd.read_csv(
    ARQUIVO_TEMP_ID,
    sep=SEPARADOR,
    encoding="utf-8-sig",
    usecols=["ID_GLEBA"],
    chunksize=TAMANHO_BLOCO,
    low_memory=False
)

for bloco in leitor_validacao:

    ids = bloco["ID_GLEBA"]

    inicio_esperado = id_esperado
    fim_esperado = id_esperado + len(bloco) - 1

    if ids.iloc[0] != inicio_esperado:
        raise RuntimeError(
            f"Falha na sequência: esperado ID {inicio_esperado}, "
            f"mas foi encontrado {ids.iloc[0]}."
        )

    if ids.iloc[-1] != fim_esperado:
        raise RuntimeError(
            f"Falha na sequência: esperado ID {fim_esperado}, "
            f"mas foi encontrado {ids.iloc[-1]}."
        )

    if ids.duplicated().any():
        raise RuntimeError(
            "Foram encontrados ID_GLEBA duplicados."
        )

    total_validado += len(bloco)
    id_esperado = fim_esperado + 1

    del bloco, ids
    gc.collect()

if total_validado != total_processado:
    raise RuntimeError(
        f"Total processado: {total_processado:,}. "
        f"Total validado: {total_validado:,}."
    )

print("Validação aprovada.")
print(f"Primeiro ID_GLEBA: 1")
print(f"Último ID_GLEBA: {total_validado:,}")

Validação aprovada.
Primeiro ID_GLEBA: 1
Último ID_GLEBA: 480,679


In [19]:
# ============================================================
# PRESERVAR O ORIGINAL E SUBSTITUIR PELO ARQUIVO COM ID_GLEBA
# ============================================================

# O backup sem ID_GLEBA é criado somente na primeira execução.
if not ARQUIVO_BACKUP_SEM_ID.exists():
    shutil.copy2(
        ARQUIVO_FINAL,
        ARQUIVO_BACKUP_SEM_ID
    )

    print("Arquivo original preservado em:")
    print(ARQUIVO_BACKUP_SEM_ID)
else:
    print("O backup original já existe:")
    print(ARQUIVO_BACKUP_SEM_ID)

# Substitui o consolidado somente depois da validação.
os.replace(
    ARQUIVO_TEMP_ID,
    ARQUIVO_FINAL
)

print("\nArquivo final atualizado com sucesso:")
print(ARQUIVO_FINAL)

# Conferência visual.
df_conferencia_id = pd.read_csv(
    ARQUIVO_FINAL,
    sep=SEPARADOR,
    encoding="utf-8-sig",
    nrows=5,
    low_memory=False
)

print("\nPrimeiros registros:")
display(df_conferencia_id)

Arquivo original preservado em:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/segundo/sicor_cerrado_2021_2025/sicor_cerrado_2021_2025_SEM_ID_GLEBA.csv

Arquivo final atualizado com sucesso:
/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/segundo/sicor_cerrado_2021_2025/sicor_cerrado_2021_2025.csv

Primeiros registros:


,ID_GLEBA,#REF_BACEN,NU_ORDEM,NU_INDICE,GT_GEOMETRIA,ANO_BASE
0,1,511405404,1,0,"POLYGON ((-47.521674 -18.527653 0,-47.521697 -18.528289 0,-47.519128 -18.528105 0,-47.519506 -18.526262 0,-47.519955...",2021
1,2,511405405,1,0,"POLYGON ((-47.579356 -18.906067 0,-47.578942 -18.906589 0,-47.578871 -18.907144 0,-47.578482 -18.907021 0,-47.57761 ...",2021
2,3,511405424,1,0,"POLYGON ((-44.441404 -16.832015 0,-44.441419 -16.831524 0,-44.440824 -16.831596 0,-44.439808999999897 -16.831699 0,-...",2021
3,4,511405446,1,0,"POLYGON ((-44.965694 -20.079657 0,-44.967058 -20.077494 0,-44.968458 -20.078419 0,-44.970597 -20.078106 0,-44.972607...",2021
4,5,511405448,1,0,"POLYGON ((-45.075154 -18.327891 0,-45.077628 -18.329547 0,-45.078869 -18.329854 0,-45.079659 -18.329899 0,-45.08048 ...",2021
